In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
from sklearn.metrics import r2_score
from matplotlib.lines import Line2D
from scipy.fft import fft, fftfreq
import xarray as xr
from openpiv import tools, pyprocess, validation, filters, scaling

sys.path.append('/Users/kjesta/Desktop/LABDATA/')
sys.path.append('/Users/kjesta/Desktop/Master prosjekt/Master_code/Lab/')
sys.path.append('/Users/kjesta/Desktop/Master prosjekt/Master_code/Lab/Probe_analysis')

from funcs import *
from processing_funcs import *
from analysis_funcs import *

import matplotlib.style as mplstyle
mplstyle.use(["ggplot", "fast"])

import warnings
warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

# For plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.style as mplstyle
mplstyle.use(['ggplot', 'fast'])

mpl.rcParams['figure.dpi'] = 200
mpl.rcParams['font.size'] = 12
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12

from matplotlib.transforms import ScaledTranslation
from matplotlib.colors import LinearSegmentedColormap
from cmocean import cm as cmo
import cmcrameri.cm as cm
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
from matplotlib.colors import Normalize

In [2]:
f14a03r1 = xr.open_dataset('/Users/kjesta/Desktop/LABDATA/PIV/Results/zoom/Final/f14a03_rec_res_r1.nc')
f14a03r2 = xr.open_dataset('/Users/kjesta/Desktop/LABDATA/PIV/Results/zoom/Final/f14a03_rec_res_r2.nc')
f14a03r3 = xr.open_dataset('/Users/kjesta/Desktop/LABDATA/PIV/Results/zoom/Final/f14a03_rec_res_r3.nc')

f14a02r1 = xr.open_dataset('/Users/kjesta/Desktop/LABDATA/PIV/Results/zoom/Final/f14a02_rec_res_r1.nc')
f14a02r2 = xr.open_dataset('/Users/kjesta/Desktop/LABDATA/PIV/Results/zoom/Final/f14a02_rec_res_r2.nc')
f14a02r3 = xr.open_dataset('/Users/kjesta/Desktop/LABDATA/PIV/Results/zoom/Final/f14a02_rec_res_r3.nc')

f14a01r1 = xr.open_dataset('/Users/kjesta/Desktop/LABDATA/PIV/Results/zoom/Final/f14a01_rec_res_r1.nc')
f14a01r2 = xr.open_dataset('/Users/kjesta/Desktop/LABDATA/PIV/Results/zoom/Final/f14a01_rec_res_r2.nc')
f14a01r3 = xr.open_dataset('/Users/kjesta/Desktop/LABDATA/PIV/Results/zoom/Final/f14a01_rec_res_r3.nc')

In [3]:
# Opening frames needed for analyzing two wave periods
ims = np.linspace(1250, 1516, 267)  # Adjusted to cover three periods (1250 to 1516)
frames_a03 = {}
frames_a02 = {}
frames_a01 = {}

for im in ims:
    im_int = int(im)
    key = 10 + im_int * 0.008
    img01 = tools.imread(f'/Users/kjesta/Desktop/LABDATA/PIV/f14a01/bitmap3/image_p{im_int}.bmp')
    frames_a01[key] = img01
    img02 = tools.imread(f'/Users/kjesta/Desktop/LABDATA/PIV/f14a02/bitmap3/image_p{im_int}.bmp')
    frames_a02[key] = img02
    img03 = tools.imread(f'/Users/kjesta/Desktop/LABDATA/PIV/f14a03/bitmap3/image_p{im_int}.bmp')
    frames_a03[key] = img03


y_max = 1490
x_min = 15
x_max = 1250

# Crop the frames to the region of interest, but keep the top part to show wave phase
for key in frames_a03.keys():
    frames_a03[key] = frames_a03[key][:y_max, x_min:x_max]
for key in frames_a02.keys():
    frames_a02[key] = frames_a02[key][:y_max, x_min:x_max]
for key in frames_a01.keys():
    frames_a01[key] = frames_a01[key][:y_max, x_min:x_max]


In [4]:
# Keep keys that are in the datasets
keys_a03 = list(frames_a03.keys())[::3]
keys_a02 = list(frames_a02.keys())[::5]
keys_a01 = list(frames_a01.keys())[::7]

frames_a03 = {k: frames_a03[k] for k in keys_a03}
frames_a02 = {k: frames_a02[k] for k in keys_a02}
frames_a01 = {k: frames_a01[k] for k in keys_a01}

trough = f14a03r1.time.values[13]
mid = f14a03r1.time.values[20]
crest = f14a03r1.time.values[27]

def nearest_idx(time_array, target_time):
    return int(np.argmin(np.abs(time_array - target_time)))

In [5]:
t_a03 = f14a03r1.time.values
trough_idx_a03 = nearest_idx(t_a03, trough)
mid_idx_a03    = nearest_idx(t_a03, mid)
crest_idx_a03  = nearest_idx(t_a03, crest)

trough_frame_a03 = list(frames_a03.keys())[trough_idx_a03]
mid_frame_a03    = list(frames_a03.keys())[mid_idx_a03]
crest_frame_a03  = list(frames_a03.keys())[crest_idx_a03]

t_a02 = f14a02r1.time.values
trough_idx_a02 = nearest_idx(t_a02, trough)
mid_idx_a02    = nearest_idx(t_a02, mid)
crest_idx_a02  = nearest_idx(t_a02, crest)

trough_frame_a02 = list(frames_a02.keys())[trough_idx_a02]
mid_frame_a02    = list(frames_a02.keys())[mid_idx_a02]
crest_frame_a02  = list(frames_a02.keys())[crest_idx_a02]

t_a01 = f14a01r1.time.values
trough_idx_a01 = nearest_idx(t_a01, trough)
mid_idx_a01    = nearest_idx(t_a01, mid)
crest_idx_a01  = nearest_idx(t_a01, crest)

trough_frame_a01 = list(frames_a01.keys())[trough_idx_a01]
mid_frame_a01    = list(frames_a01.keys())[mid_idx_a01]
crest_frame_a01  = list(frames_a01.keys())[crest_idx_a01]

In [6]:
x = f14a03r1.x.values
z = f14a03r1.z.values
x_max = x.max()

In [7]:
def depth_formatter(y_tick, pos):
        # depth: negative downward
        # Adding ashift the depth values so that it matches real water depth
        return f"{-(y_tick + 0.09):.2f}"

def x_phys_formatter(x_tick, pos):
        # Map image x → physical x (paddle at 8.06 m on the right)
        return f"{8.06 + (x_max - x_tick):.2f}"

In [10]:
def plot_velocity_fields(point):
    if point == 'trough':
        time_idx_a03 = trough_idx_a03
        time_idx_a02 = trough_idx_a02
        time_idx_a01 = trough_idx_a01
    elif point == 'mid':
        time_idx_a03 = mid_idx_a03
        time_idx_a02 = mid_idx_a02
        time_idx_a01 = mid_idx_a01
    elif point == 'crest':
        time_idx_a03 = crest_idx_a03
        time_idx_a02 = crest_idx_a02
        time_idx_a01 = crest_idx_a01
    kwargs = {'format': '%.3f'}

    fig, ax = plt.subplots(
        4, 3,
        figsize=(12, 16),
        sharex=True,
        gridspec_kw={'height_ratios': [0.5, 1, 1, 1]}  # top row half the height
    )

    umin = 0
    umax = 0.03

    # Subsampling steps for quiver plot
    step_x = 6
    step_z = 4

    # Images in top row
    ax[0, 0].imshow(
        frames_a03[list(frames_a03.keys())[time_idx_a03]][100:650, :],
        extent=[x.min(), x.max(), z.min(), z.max()],
        aspect='auto',
        cmap='gray'
    )
    ax[0, 1].imshow(
        frames_a02[list(frames_a02.keys())[time_idx_a02]][100:650, :],
        extent=[x.min(), x.max(), z.min(), z.max()],
        aspect='auto',
        cmap='gray'
    )
    ax[0, 2].imshow(
        frames_a01[list(frames_a01.keys())[time_idx_a01]][100:650, :],
        extent=[x.min(), x.max(), z.min(), z.max()],
        aspect='auto',
        cmap='gray'
    )

    for a in ax[0, :]:
        a.tick_params(
            bottom=False, labelbottom=False,
            left=False,   labelleft=False
        )
        a.grid()

    # velocity fields in rows 1–3
    im = ax[1, 0].contourf(
        x, z,
        np.sqrt(
            f14a03r1.u.values[time_idx_a03, :, :]**2 +
            f14a03r1.w.values[time_idx_a03, :, :]**2
        ),
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.speed
    )
    ax[1, 0].quiver(
        x[::step_x], z[::step_z],
        -f14a03r1.u.values[time_idx_a03, ::step_z, ::step_x],
        f14a03r1.w.values[time_idx_a03, ::step_z, ::step_x]
    )

    ax[2, 0].contourf(
        x, z,
        np.sqrt(
            f14a03r2.u.values[time_idx_a03, :, :]**2 +
            f14a03r2.w.values[time_idx_a03, :, :]**2
        ),
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.speed
    )
    ax[2, 0].quiver(
        x[::step_x], z[::step_z],
        -f14a03r2.u.values[time_idx_a03, ::step_z, ::step_x],
        f14a03r2.w.values[time_idx_a03, ::step_z, ::step_x]
    )

    ax[3, 0].contourf(
        x, z,
        np.sqrt(
            f14a03r3.u.values[time_idx_a03, :, :]**2 +
            f14a03r3.w.values[time_idx_a03, :, :]**2
        ),
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.speed
    )
    ax[3, 0].quiver(
        x[::step_x], z[::step_z],
        -f14a03r3.u.values[time_idx_a03, ::step_z, ::step_x],
        f14a03r3.w.values[time_idx_a03, ::step_z, ::step_x]
    )

    ax[1, 1].contourf(
        x, z,
        np.sqrt(
            f14a02r1.u.values[time_idx_a02, :, :]**2 +
            f14a02r1.w.values[time_idx_a02, :, :]**2
        ),
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.speed
    )
    ax[1, 1].quiver(
        x[::step_x], z[::step_z],
        -f14a02r1.u.values[time_idx_a02, ::step_z, ::step_x],
        f14a02r1.w.values[time_idx_a02, ::step_z, ::step_x]
    )

    ax[2, 1].contourf(
        x, z,
        np.sqrt(
            f14a02r2.u.values[time_idx_a02, :, :]**2 +
            f14a02r2.w.values[time_idx_a02, :, :]**2
        ),
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.speed
    )
    ax[2, 1].quiver(
        x[::step_x], z[::step_z],
        -f14a02r2.u.values[time_idx_a02, ::step_z, ::step_x],
        f14a02r2.w.values[time_idx_a02, ::step_z, ::step_x]
    )

    ax[3, 1].contourf(
        x, z,
        np.sqrt(
            f14a02r3.u.values[time_idx_a02, :, :]**2 +
            f14a02r3.w.values[time_idx_a02, :, :]**2
        ),
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.speed
    )
    ax[3, 1].quiver(
        x[::step_x], z[::step_z],
        -f14a02r3.u.values[time_idx_a02, ::step_z, ::step_x],
        f14a02r3.w.values[time_idx_a02, ::step_z, ::step_x]
    )

    ax[1, 2].contourf(
        x, z,
        np.sqrt(
            f14a01r1.u.values[time_idx_a01, :, :]**2 +
            f14a01r1.w.values[time_idx_a01, :, :]**2
        ),
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.speed
    )
    ax[1, 2].quiver(
        x[::step_x], z[::step_z],
        -f14a01r1.u.values[time_idx_a01, ::step_z, ::step_x],
        f14a01r1.w.values[time_idx_a01, ::step_z, ::step_x]
    )

    ax[2, 2].contourf(
        x, z,
        np.sqrt(
            f14a01r2.u.values[time_idx_a01, :, :]**2 +
            f14a01r2.w.values[time_idx_a01, :, :]**2
        ),
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.speed
    )
    ax[2, 2].quiver(
        x[::step_x], z[::step_z],
        -f14a01r2.u.values[time_idx_a01, ::step_z, ::step_x],
        f14a01r2.w.values[time_idx_a01, ::step_z, ::step_x]
    )

    ax[3, 2].contourf(
        x, z,
        np.sqrt(
            f14a01r3.u.values[time_idx_a01, :, :]**2 +
            f14a01r3.w.values[time_idx_a01, :, :]**2
        ),
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.speed
    )
    ax[3, 2].quiver(
        x[::step_x], z[::step_z],
        -f14a01r3.u.values[time_idx_a01, ::step_z, ::step_x],
        f14a01r3.w.values[time_idx_a01, ::step_z, ::step_x]
    )

    ax[0, 0].set_title(
        fr"$\mathbf{{A = 0.3V,\, t = {f14a03r1['time'][time_idx_a03]:.3f}\, s}}$"
        "\nRun 1"
    )
    ax[1, 0].set_title("Run 1")
    ax[2, 0].set_title("Run 2")
    ax[3, 0].set_title("Run 3")

    ax[0, 1].set_title(
        fr"$\mathbf{{A = 0.2V,\, t = {f14a02r1['time'][time_idx_a02]:.3f}\, s}}$"
        "\nRun 1"
    )
    ax[1, 1].set_title("Run 1")
    ax[2, 1].set_title("Run 2")
    ax[3, 1].set_title("Run 3")

    ax[0, 2].set_title(
        fr"$\mathbf{{A = 0.1V,\, t = {f14a01r1['time'][time_idx_a01]:.3f}\, s}}$"
        "\nRun 1"
    )
    ax[1, 2].set_title("Run 1")
    ax[2, 2].set_title("Run 2")
    ax[3, 2].set_title("Run 3")

    for a in ax[1:, :].flatten():
        a.yaxis.set_major_formatter(FuncFormatter(depth_formatter))
        a.invert_yaxis()
        a.hlines(
            z[42],
            xmin=x.min(),
            xmax=x.max(),
            colors='white',
            linestyles='dashed',
            linewidth=2,
        )

    for col in [1, 2]:
        for row in range(1, 4):
            ax[row, col].tick_params(left=False, labelleft=False)
    for row in [1, 2]:
        for col in range(3):
            ax[row, col].tick_params(bottom=False, labelbottom=False)
    
    ax[3, 0].xaxis.set_major_formatter(FuncFormatter(x_phys_formatter))
    ax[3, 1].xaxis.set_major_formatter(FuncFormatter(x_phys_formatter))
    ax[3, 2].xaxis.set_major_formatter(FuncFormatter(x_phys_formatter))

    ax[1, 0].set_ylabel("z [m]")
    ax[2, 0].set_ylabel("z [m]")
    ax[3, 0].set_ylabel("z [m]")

    ax[3, 0].set_xlabel("x [m]")
    ax[3, 1].set_xlabel("x [m]")
    ax[3, 2].set_xlabel("x [m]")

    cbar = fig.colorbar(
        im,
        ax=ax,  # or ax=ax[1:, :].ravel() if you want only lower rows
        orientation='horizontal',
        location="bottom",
        extend='max',
        fraction=0.046,
        pad=0.04,
        **kwargs
    )
    cbar.set_label("Velocity magnitude [m/s]")

    plt.savefig(
        f'/Users/kjesta/Desktop/Master prosjekt/Figurer/PIV/matched/velocity_fields_{time_idx_a01}.png',
        dpi=200
    )
    plt.close(fig)

In [11]:
plot_velocity_fields('trough')
plot_velocity_fields('mid')
plot_velocity_fields('crest')

In [12]:
def plot_vorticity_fields(point):
    if point == 'trough':
        time_idx_a03 = trough_idx_a03
        time_idx_a02 = trough_idx_a02
        time_idx_a01 = trough_idx_a01

    elif point == 'mid':
        time_idx_a03 = mid_idx_a03
        time_idx_a02 = mid_idx_a02
        time_idx_a01 = mid_idx_a01

    elif point == 'crest':
        time_idx_a03 = crest_idx_a03
        time_idx_a02 = crest_idx_a02
        time_idx_a01 = crest_idx_a01

    x_max = x.max()
    kwargs = {'format': '%.3f'}

    fig, ax = plt.subplots(
        4, 3,
        figsize=(12, 16),
        sharex=True,
        gridspec_kw={'height_ratios': [0.5, 1, 1, 1]}  # top row half the height
    )

    # Colorbar limits for vorticity
    umin = -0.004
    umax = 0.004

    # Images in top row
    ax[0, 0].imshow(
        frames_a03[list(frames_a03.keys())[time_idx_a03]][100:650, :],
        extent=[x.min(), x.max(), z.min(), z.max()],
        aspect='auto',
        cmap='gray'
    )
    ax[0, 1].imshow(
        frames_a02[list(frames_a02.keys())[time_idx_a02]][100:650, :],
        extent=[x.min(), x.max(), z.min(), z.max()],
        aspect='auto',
        cmap='gray'
    )
    ax[0, 2].imshow(
        frames_a01[list(frames_a01.keys())[time_idx_a01]][100:650, :],
        extent=[x.min(), x.max(), z.min(), z.max()],
        aspect='auto',
        cmap='gray'
    )

    for a in ax[0, :]:
        a.tick_params(
            bottom=False, labelbottom=False,
            left=False,   labelleft=False
        )
        a.grid()

    # vorticity fields in rows 1–3
    im = ax[1, 0].contourf(
        x, z,
        f14a03r1['vorticity'][time_idx_a03, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.curl
    )
    ax[2, 0].contourf(
        x, z,
        f14a03r2['vorticity'][time_idx_a03, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.curl
    )
    ax[3, 0].contourf(
        x, z,
        f14a03r3['vorticity'][time_idx_a03, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.curl
    )

    ax[1, 1].contourf(
        x, z,
        f14a02r1['vorticity'][time_idx_a02, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.curl
    )
    ax[2, 1].contourf(
        x, z,
        f14a02r2['vorticity'][time_idx_a02, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.curl
    )
    ax[3, 1].contourf(
        x, z,
        f14a02r3['vorticity'][time_idx_a02, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.curl
    )

    ax[1, 2].contourf(
        x, z,
        f14a01r1['vorticity'][time_idx_a01, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.curl
    )
    ax[2, 2].contourf(
        x, z,
        f14a01r2['vorticity'][time_idx_a01, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.curl
    )
    ax[3, 2].contourf(
        x, z,
        f14a01r3['vorticity'][time_idx_a01, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.curl
    )

    ax[0, 0].set_title(
        fr"$\mathbf{{A = 0.3V,\, t = {f14a03r1['time'][time_idx_a03]:.3f}\, s}}$"
        "\nRun 1"
    )
    ax[1, 0].set_title("Run 1")
    ax[2, 0].set_title("Run 2")
    ax[3, 0].set_title("Run 3")

    ax[0, 1].set_title(
        fr"$\mathbf{{A = 0.2V,\, t = {f14a02r1['time'][time_idx_a02]:.3f}\, s}}$"
        "\nRun 1"
    )
    ax[1, 1].set_title("Run 1")
    ax[2, 1].set_title("Run 2")
    ax[3, 1].set_title("Run 3")

    ax[0, 2].set_title(
        fr"$\mathbf{{A = 0.1V,\, t = {f14a01r1['time'][time_idx_a01]:.3f}\, s}}$"
        "\nRun 1"
    )
    ax[1, 2].set_title("Run 1")
    ax[2, 2].set_title("Run 2")
    ax[3, 2].set_title("Run 3")

    for a in ax[1:, :].flatten():
        a.yaxis.set_major_formatter(FuncFormatter(depth_formatter))
        a.invert_yaxis()
        a.hlines(
            z[42],
            xmin=x.min(),
            xmax=x.max(),
            colors='white',
            linestyles='dashed',
            linewidth=2,
        )

    ax[3, 0].xaxis.set_major_formatter(FuncFormatter(x_phys_formatter))
    ax[3, 1].xaxis.set_major_formatter(FuncFormatter(x_phys_formatter))
    ax[3, 2].xaxis.set_major_formatter(FuncFormatter(x_phys_formatter))
    
    for col in [1, 2]:
        for row in range(1, 4):
            ax[row, col].tick_params(left=False, labelleft=False)
    for row in [1, 2]:
        for col in range(3):
            ax[row, col].tick_params(bottom=False, labelbottom=False)

    ax[1, 0].set_ylabel("z [m]")
    ax[2, 0].set_ylabel("z [m]")
    ax[3, 0].set_ylabel("z [m]")

    ax[3, 0].set_xlabel("x [m]")
    ax[3, 1].set_xlabel("x [m]")
    ax[3, 2].set_xlabel("x [m]")

    cbar = fig.colorbar(
        im,
        ax=ax,  # or ax=ax[1:, :].ravel() if you want only lower rows
        orientation='horizontal',
        location="bottom",
        extend='max',
        fraction=0.046,
        pad=0.04,
        **kwargs
    )
    cbar.set_label("Vorticity [1/s]")

    plt.savefig(
        f'/Users/kjesta/Desktop/Master prosjekt/Figurer/PIV/matched/vorticity_fields_{time_idx_a03}.png',
        dpi=200)
    plt.close(fig)

In [13]:
plot_vorticity_fields('trough')
plot_vorticity_fields('mid')
plot_vorticity_fields('crest')

In [14]:
def plot_vertical_velocity_fields(point):
    if point == 'trough':
        time_idx_a03 = trough_idx_a03
        time_idx_a02 = trough_idx_a02
        time_idx_a01 = trough_idx_a01

    elif point == 'mid':
        time_idx_a03 = mid_idx_a03
        time_idx_a02 = mid_idx_a02
        time_idx_a01 = mid_idx_a01

    elif point == 'crest':
        time_idx_a03 = crest_idx_a03
        time_idx_a02 = crest_idx_a02
        time_idx_a01 = crest_idx_a01

    x_max = x.max()
    kwargs = {'format': '%.3f'}

    fig, ax = plt.subplots(
        4, 3,
        figsize=(12, 16),
        sharex=True,
        gridspec_kw={'height_ratios': [0.5, 1, 1, 1]}  # top row half the height
    )

    # Colorbar limits for vertical velocity
    umin = -0.02
    umax = 0.02

    # Images in top row
    ax[0, 0].imshow(
        frames_a03[list(frames_a03.keys())[time_idx_a03]][100:650, :],
        extent=[x.min(), x.max(), z.min(), z.max()],
        aspect='auto',
        cmap='gray'
    )
    ax[0, 1].imshow(
        frames_a02[list(frames_a02.keys())[time_idx_a02]][100:650, :],
        extent=[x.min(), x.max(), z.min(), z.max()],
        aspect='auto',
        cmap='gray'
    )
    ax[0, 2].imshow(
        frames_a01[list(frames_a01.keys())[time_idx_a01]][100:650, :],
        extent=[x.min(), x.max(), z.min(), z.max()],
        aspect='auto',
        cmap='gray'
    )

    for a in ax[0, :]:
        a.tick_params(
            bottom=False, labelbottom=False,
            left=False,   labelleft=False
        )
        a.grid()

    # vertical velocity fields in rows 1–3
    im = ax[1, 0].contourf(
        x, z,
        f14a03r1['w'][time_idx_a03, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.balance
    )
    ax[2, 0].contourf(
        x, z,
        f14a03r2['w'][time_idx_a03, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.balance
    )
    ax[3, 0].contourf(
        x, z,
        f14a03r3['w'][time_idx_a03, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.balance
    )

    ax[1, 1].contourf(
        x, z,
        f14a02r1['w'][time_idx_a02, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.balance
    )
    ax[2, 1].contourf(
        x, z,
        f14a02r2['w'][time_idx_a02, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.balance
    )
    ax[3, 1].contourf(
        x, z,
        f14a02r3['w'][time_idx_a02, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.balance
    )

    ax[1, 2].contourf(
        x, z,
        f14a01r1['w'][time_idx_a01, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.balance
    )
    ax[2, 2].contourf(
        x, z,
        f14a01r2['w'][time_idx_a01, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.balance
    )
    ax[3, 2].contourf(
        x, z,
        f14a01r3['w'][time_idx_a01, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.balance
    )

    ax[0, 0].set_title(
        fr"$\mathbf{{A = 0.3V,\, t = {f14a03r1['time'][time_idx_a03]:.3f}\, s}}$"
        "\nRun 1"
    )
    ax[1, 0].set_title("Run 1")
    ax[2, 0].set_title("Run 2")
    ax[3, 0].set_title("Run 3")

    ax[0, 1].set_title(
        fr"$\mathbf{{A = 0.2V,\, t = {f14a02r1['time'][time_idx_a02]:.3f}\, s}}$"
        "\nRun 1"
    )
    ax[1, 1].set_title("Run 1")
    ax[2, 1].set_title("Run 2")
    ax[3, 1].set_title("Run 3")

    ax[0, 2].set_title(
        fr"$\mathbf{{A = 0.1V,\, t = {f14a01r1['time'][time_idx_a01]:.3f}\, s}}$"
        "\nRun 1"
    )
    ax[1, 2].set_title("Run 1")
    ax[2, 2].set_title("Run 2")
    ax[3, 2].set_title("Run 3")

    for a in ax[1:, :].flatten():
        a.yaxis.set_major_formatter(FuncFormatter(depth_formatter))
        a.invert_yaxis()
        a.hlines(
            z[42],
            xmin=x.min(),
            xmax=x.max(),
            colors='white',
            linestyles='dashed',
            linewidth=2,
        )

    ax[3, 0].xaxis.set_major_formatter(FuncFormatter(x_phys_formatter))
    ax[3, 1].xaxis.set_major_formatter(FuncFormatter(x_phys_formatter))
    ax[3, 2].xaxis.set_major_formatter(FuncFormatter(x_phys_formatter))

    for col in [1, 2]:
        for row in range(1, 4):
            ax[row, col].tick_params(left=False, labelleft=False)
    for row in [1, 2]:
        for col in range(3):
            ax[row, col].tick_params(bottom=False, labelbottom=False)

    ax[1, 0].set_ylabel("z [m]")
    ax[2, 0].set_ylabel("z [m]")
    ax[3, 0].set_ylabel("z [m]")

    ax[3, 0].set_xlabel("x [m]")
    ax[3, 1].set_xlabel("x [m]")
    ax[3, 2].set_xlabel("x [m]")

    cbar = fig.colorbar(
        im,
        ax=ax,  # or ax=ax[1:, :].ravel() if you want only lower rows
        orientation='horizontal',
        location="bottom",
        extend='max',
        fraction=0.046,
        pad=0.04,
        **kwargs
    )
    cbar.set_label("Vertical Velocity [m/s]")

    plt.savefig(
        f'/Users/kjesta/Desktop/Master prosjekt/Figurer/PIV/matched/vertical_fields_{time_idx_a03}.png',
        dpi=200
    )
    plt.close(fig)

In [15]:
plot_vertical_velocity_fields('trough')
plot_vertical_velocity_fields('mid')
plot_vertical_velocity_fields('crest')

In [48]:
def plot_velocity_fields(time_idx):
    kwargs = {'format': '%.3f'}

    fig, ax = plt.subplots(
        4, 3,
        figsize=(12, 16),
        sharex=True,
        gridspec_kw={'height_ratios': [0.5, 1, 1, 1]}  # top row half the height
    )

    umin = 0
    umax = 0.03

    # Subsampling steps for quiver plot
    step_x = 6
    step_z = 4

    # Images in top row
    ax[0, 0].imshow(
        frames_a03[list(frames_a03.keys())[time_idx]][100:650, :],
        extent=[x.min(), x.max(), z.min(), z.max()],
        aspect='auto',
        cmap='gray'
    )
    ax[0, 1].imshow(
        frames_a02[list(frames_a02.keys())[time_idx]][100:650, :],
        extent=[x.min(), x.max(), z.min(), z.max()],
        aspect='auto',
        cmap='gray'
    )
    ax[0, 2].imshow(
        frames_a01[list(frames_a01.keys())[time_idx]][100:650, :],
        extent=[x.min(), x.max(), z.min(), z.max()],
        aspect='auto',
        cmap='gray'
    )

    for a in ax[0, :]:
        a.tick_params(
            bottom=False, labelbottom=False,
            left=False,   labelleft=False
        )
        a.grid()

    # velocity fields in rows 1–3
    im = ax[1, 0].contourf(
        x, z,
        np.sqrt(
            f14a03r1.u.values[time_idx, :, :]**2 +
            f14a03r1.w.values[time_idx, :, :]**2
        ),
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.speed
    )
    ax[1, 0].quiver(
        x[::step_x], z[::step_z],
        -f14a03r1.u.values[time_idx, ::step_z, ::step_x],
        f14a03r1.w.values[time_idx, ::step_z, ::step_x]
    )

    ax[2, 0].contourf(
        x, z,
        np.sqrt(
            f14a03r2.u.values[time_idx, :, :]**2 +
            f14a03r2.w.values[time_idx, :, :]**2
        ),
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.speed
    )
    ax[2, 0].quiver(
        x[::step_x], z[::step_z],
        -f14a03r2.u.values[time_idx, ::step_z, ::step_x],
        f14a03r2.w.values[time_idx, ::step_z, ::step_x]
    )

    ax[3, 0].contourf(
        x, z,
        np.sqrt(
            f14a03r3.u.values[time_idx, :, :]**2 +
            f14a03r3.w.values[time_idx, :, :]**2
        ),
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.speed
    )
    ax[3, 0].quiver(
        x[::step_x], z[::step_z],
        -f14a03r3.u.values[time_idx, ::step_z, ::step_x],
        f14a03r3.w.values[time_idx, ::step_z, ::step_x]
    )

    ax[1, 1].contourf(
        x, z,
        np.sqrt(
            f14a02r1.u.values[time_idx, :, :]**2 +
            f14a02r1.w.values[time_idx, :, :]**2
        ),
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.speed
    )
    ax[1, 1].quiver(
        x[::step_x], z[::step_z],
        -f14a02r1.u.values[time_idx, ::step_z, ::step_x],
        f14a02r1.w.values[time_idx, ::step_z, ::step_x]
    )

    ax[2, 1].contourf(
        x, z,
        np.sqrt(
            f14a02r2.u.values[time_idx, :, :]**2 +
            f14a02r2.w.values[time_idx, :, :]**2
        ),
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.speed
    )
    ax[2, 1].quiver(
        x[::step_x], z[::step_z],
        -f14a02r2.u.values[time_idx, ::step_z, ::step_x],
        f14a02r2.w.values[time_idx, ::step_z, ::step_x]
    )

    ax[3, 1].contourf(
        x, z,
        np.sqrt(
            f14a02r3.u.values[time_idx, :, :]**2 +
            f14a02r3.w.values[time_idx, :, :]**2
        ),
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.speed
    )
    ax[3, 1].quiver(
        x[::step_x], z[::step_z],
        -f14a02r3.u.values[time_idx, ::step_z, ::step_x],
        f14a02r3.w.values[time_idx, ::step_z, ::step_x]
    )

    ax[1, 2].contourf(
        x, z,
        np.sqrt(
            f14a01r1.u.values[time_idx, :, :]**2 +
            f14a01r1.w.values[time_idx, :, :]**2
        ),
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.speed
    )
    ax[1, 2].quiver(
        x[::step_x], z[::step_z],
        -f14a01r1.u.values[time_idx, ::step_z, ::step_x],
        f14a01r1.w.values[time_idx, ::step_z, ::step_x]
    )

    ax[2, 2].contourf(
        x, z,
        np.sqrt(
            f14a01r2.u.values[time_idx, :, :]**2 +
            f14a01r2.w.values[time_idx, :, :]**2
        ),
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.speed
    )
    ax[2, 2].quiver(
        x[::step_x], z[::step_z],
        -f14a01r2.u.values[time_idx, ::step_z, ::step_x],
        f14a01r2.w.values[time_idx, ::step_z, ::step_x]
    )

    ax[3, 2].contourf(
        x, z,
        np.sqrt(
            f14a01r3.u.values[time_idx, :, :]**2 +
            f14a01r3.w.values[time_idx, :, :]**2
        ),
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.speed
    )
    ax[3, 2].quiver(
        x[::step_x], z[::step_z],
        -f14a01r3.u.values[time_idx, ::step_z, ::step_x],
        f14a01r3.w.values[time_idx, ::step_z, ::step_x]
    )

    ax[0, 0].set_title(
        fr"$\mathbf{{A = 0.3V,\, t = {f14a03r1['time'][time_idx]:.3f}\, s}}$"
        "\nRun 1"
    )
    ax[1, 0].set_title("Run 1")
    ax[2, 0].set_title("Run 2")
    ax[3, 0].set_title("Run 3")

    ax[0, 1].set_title(
        fr"$\mathbf{{A = 0.2V,\, t = {f14a02r1['time'][time_idx]:.3f}\, s}}$"
        "\nRun 1"
    )
    ax[1, 1].set_title("Run 1")
    ax[2, 1].set_title("Run 2")
    ax[3, 1].set_title("Run 3")

    ax[0, 2].set_title(
        fr"$\mathbf{{A = 0.1V,\, t = {f14a01r1['time'][time_idx]:.3f}\, s}}$"
        "\nRun 1"
    )
    ax[1, 2].set_title("Run 1")
    ax[2, 2].set_title("Run 2")
    ax[3, 2].set_title("Run 3")

    for a in ax[1:, :].flatten():
        a.xaxis.set_major_formatter(FuncFormatter(x_phys_formatter))
        a.yaxis.set_major_formatter(FuncFormatter(depth_formatter))
        a.invert_yaxis()
        a.hlines(
            z[42],
            xmin=x.min(),
            xmax=x.max(),
            colors='white',
            linestyles='dashed',
            linewidth=2,
        )

    for col in [1, 2]:
        for row in range(1, 4):
            ax[row, col].tick_params(left=False, labelleft=False)
    for row in [1, 2]:
        for col in range(3):
            ax[row, col].tick_params(bottom=False, labelbottom=False)

    ax[1, 0].set_ylabel("z [m]")
    ax[2, 0].set_ylabel("z [m]")
    ax[3, 0].set_ylabel("z [m]")

    ax[3, 0].set_xlabel("x [m]")
    ax[3, 1].set_xlabel("x [m]")
    ax[3, 2].set_xlabel("x [m]")

    cbar = fig.colorbar(
        im,
        ax=ax,  # or ax=ax[1:, :].ravel() if you want only lower rows
        orientation='horizontal',
        location="bottom",
        extend='max',
        fraction=0.046,
        pad=0.04,
        **kwargs
    )
    cbar.set_label("Velocity magnitude [m/s]")

    plt.savefig(
        f'/Users/kjesta/Desktop/Master prosjekt/Figurer/PIV/velocity_fields_{time_idx}.png',
        dpi=200
    )
    plt.close(fig)

In [49]:
def plot_vorticity_fields(time_idx):
    x_max = x.max()
    kwargs = {'format': '%.3f'}

    fig, ax = plt.subplots(
        4, 3,
        figsize=(12, 16),
        sharex=True,
        gridspec_kw={'height_ratios': [0.5, 1, 1, 1]}  # top row half the height
    )

    # Colorbar limits for vorticity
    umin = -0.004
    umax = 0.004

    # Images in top row
    ax[0, 0].imshow(
        frames_a03[list(frames_a03.keys())[time_idx]][100:650, :],
        extent=[x.min(), x.max(), z.min(), z.max()],
        aspect='auto',
        cmap='gray'
    )
    ax[0, 1].imshow(
        frames_a02[list(frames_a02.keys())[time_idx]][100:650, :],
        extent=[x.min(), x.max(), z.min(), z.max()],
        aspect='auto',
        cmap='gray'
    )
    ax[0, 2].imshow(
        frames_a01[list(frames_a01.keys())[time_idx]][100:650, :],
        extent=[x.min(), x.max(), z.min(), z.max()],
        aspect='auto',
        cmap='gray'
    )

    for a in ax[0, :]:
        a.tick_params(
            bottom=False, labelbottom=False,
            left=False,   labelleft=False
        )
        a.grid()

    # vorticity fields in rows 1–3
    im = ax[1, 0].contourf(
        x, z,
        f14a03r1['vorticity'][time_idx, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.curl
    )
    ax[2, 0].contourf(
        x, z,
        f14a03r2['vorticity'][time_idx, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.curl
    )
    ax[3, 0].contourf(
        x, z,
        f14a03r3['vorticity'][time_idx, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.curl
    )

    ax[1, 1].contourf(
        x, z,
        f14a02r1['vorticity'][time_idx, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.curl
    )
    ax[2, 1].contourf(
        x, z,
        f14a02r2['vorticity'][time_idx, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.curl
    )
    ax[3, 1].contourf(
        x, z,
        f14a02r3['vorticity'][time_idx, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.curl
    )

    ax[1, 2].contourf(
        x, z,
        f14a01r1['vorticity'][time_idx, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.curl
    )
    ax[2, 2].contourf(
        x, z,
        f14a01r2['vorticity'][time_idx, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.curl
    )
    ax[3, 2].contourf(
        x, z,
        f14a01r3['vorticity'][time_idx, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.curl
    )

    ax[0, 0].set_title(
        fr"$\mathbf{{A = 0.3V,\, t = {f14a03r1['time'][time_idx]:.3f}\, s}}$"
        "\nRun 1"
    )
    ax[1, 0].set_title("Run 1")
    ax[2, 0].set_title("Run 2")
    ax[3, 0].set_title("Run 3")

    ax[0, 1].set_title(
        fr"$\mathbf{{A = 0.2V,\, t = {f14a02r1['time'][time_idx]:.3f}\, s}}$"
        "\nRun 1"
    )
    ax[1, 1].set_title("Run 1")
    ax[2, 1].set_title("Run 2")
    ax[3, 1].set_title("Run 3")

    ax[0, 2].set_title(
        fr"$\mathbf{{A = 0.1V,\, t = {f14a01r1['time'][time_idx]:.3f}\, s}}$"
        "\nRun 1"
    )
    ax[1, 2].set_title("Run 1")
    ax[2, 2].set_title("Run 2")
    ax[3, 2].set_title("Run 3")

    for a in ax[1:, :].flatten():
        a.xaxis.set_major_formatter(FuncFormatter(x_phys_formatter))
        a.yaxis.set_major_formatter(FuncFormatter(depth_formatter))
        a.invert_yaxis()
        a.hlines(
            z[42],
            xmin=x.min(),
            xmax=x.max(),
            colors='white',
            linestyles='dashed',
            linewidth=2,
        )

    for col in [1, 2]:
        for row in range(1, 4):
            ax[row, col].tick_params(left=False, labelleft=False)
    for row in [1, 2]:
        for col in range(3):
            ax[row, col].tick_params(bottom=False, labelbottom=False)

    ax[1, 0].set_ylabel("z [m]")
    ax[2, 0].set_ylabel("z [m]")
    ax[3, 0].set_ylabel("z [m]")

    ax[3, 0].set_xlabel("x [m]")
    ax[3, 1].set_xlabel("x [m]")
    ax[3, 2].set_xlabel("x [m]")

    cbar = fig.colorbar(
        im,
        ax=ax,  # or ax=ax[1:, :].ravel() if you want only lower rows
        orientation='horizontal',
        location="bottom",
        extend='max',
        fraction=0.046,
        pad=0.04,
        **kwargs
    )
    cbar.set_label("Vorticity [1/s]")

    plt.savefig(
        f'/Users/kjesta/Desktop/Master prosjekt/Figurer/PIV/vorticity_fields_{time_idx}.png',
        dpi=200)
    plt.close(fig)

In [50]:
def plot_vertical_velocity_fields(time_idx):
    x_max = x.max()
    kwargs = {'format': '%.3f'}

    fig, ax = plt.subplots(
        4, 3,
        figsize=(12, 16),
        sharex=True,
        gridspec_kw={'height_ratios': [0.5, 1, 1, 1]}  # top row half the height
    )

    # Colorbar limits for vertical velocity
    umin = -0.02
    umax = 0.02

    # Images in top row
    ax[0, 0].imshow(
        frames_a03[list(frames_a03.keys())[time_idx]][100:650, :],
        extent=[x.min(), x.max(), z.min(), z.max()],
        aspect='auto',
        cmap='gray'
    )
    ax[0, 1].imshow(
        frames_a02[list(frames_a02.keys())[time_idx]][100:650, :],
        extent=[x.min(), x.max(), z.min(), z.max()],
        aspect='auto',
        cmap='gray'
    )
    ax[0, 2].imshow(
        frames_a01[list(frames_a01.keys())[time_idx]][100:650, :],
        extent=[x.min(), x.max(), z.min(), z.max()],
        aspect='auto',
        cmap='gray'
    )

    for a in ax[0, :]:
        a.tick_params(
            bottom=False, labelbottom=False,
            left=False,   labelleft=False
        )
        a.grid()

    # vertical velocity fields in rows 1–3
    im = ax[1, 0].contourf(
        x, z,
        f14a03r1['w'][time_idx, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.balance
    )
    ax[2, 0].contourf(
        x, z,
        f14a03r2['w'][time_idx, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.balance
    )
    ax[3, 0].contourf(
        x, z,
        f14a03r3['w'][time_idx, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.balance
    )

    ax[1, 1].contourf(
        x, z,
        f14a02r1['w'][time_idx, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.balance
    )
    ax[2, 1].contourf(
        x, z,
        f14a02r2['w'][time_idx, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.balance
    )
    ax[3, 1].contourf(
        x, z,
        f14a02r3['w'][time_idx, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.balance
    )

    ax[1, 2].contourf(
        x, z,
        f14a01r1['w'][time_idx, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.balance
    )
    ax[2, 2].contourf(
        x, z,
        f14a01r2['w'][time_idx, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.balance
    )
    ax[3, 2].contourf(
        x, z,
        f14a01r3['w'][time_idx, :, :],
        levels=np.linspace(umin, umax, 41),
        cmap=cmo.balance
    )

    ax[0, 0].set_title(
        fr"$\mathbf{{A = 0.3V,\, t = {f14a03r1['time'][time_idx]:.3f}\, s}}$"
        "\nRun 1"
    )
    ax[1, 0].set_title("Run 1")
    ax[2, 0].set_title("Run 2")
    ax[3, 0].set_title("Run 3")

    ax[0, 1].set_title(
        fr"$\mathbf{{A = 0.2V,\, t = {f14a02r1['time'][time_idx]:.3f}\, s}}$"
        "\nRun 1"
    )
    ax[1, 1].set_title("Run 1")
    ax[2, 1].set_title("Run 2")
    ax[3, 1].set_title("Run 3")

    ax[0, 2].set_title(
        fr"$\mathbf{{A = 0.1V,\, t = {f14a01r1['time'][time_idx]:.3f}\, s}}$"
        "\nRun 1"
    )
    ax[1, 2].set_title("Run 1")
    ax[2, 2].set_title("Run 2")
    ax[3, 2].set_title("Run 3")

    for a in ax[1:, :].flatten():
        a.xaxis.set_major_formatter(FuncFormatter(x_phys_formatter))
        a.yaxis.set_major_formatter(FuncFormatter(depth_formatter))
        a.invert_yaxis()
        a.hlines(
            z[42],
            xmin=x.min(),
            xmax=x.max(),
            colors='white',
            linestyles='dashed',
            linewidth=2,
        )

    for col in [1, 2]:
        for row in range(1, 4):
            ax[row, col].tick_params(left=False, labelleft=False)
    for row in [1, 2]:
        for col in range(3):
            ax[row, col].tick_params(bottom=False, labelbottom=False)

    ax[1, 0].set_ylabel("z [m]")
    ax[2, 0].set_ylabel("z [m]")
    ax[3, 0].set_ylabel("z [m]")

    ax[3, 0].set_xlabel("x [m]")
    ax[3, 1].set_xlabel("x [m]")
    ax[3, 2].set_xlabel("x [m]")

    cbar = fig.colorbar(
        im,
        ax=ax,  # or ax=ax[1:, :].ravel() if you want only lower rows
        orientation='horizontal',
        location="bottom",
        extend='max',
        fraction=0.046,
        pad=0.04,
        **kwargs
    )
    cbar.set_label("Vertical Velocity [m/s]")

    plt.savefig(
        f'/Users/kjesta/Desktop/Master prosjekt/Figurer/PIV/vertical_fields_{time_idx}.png',
        dpi=200
    )
    plt.close(fig)

In [51]:
for i in range(0, 36):
    plot_velocity_fields(i)
    plot_vorticity_fields(i)
    plot_vertical_velocity_fields(i)